In [1]:
initial_prompt="""**Task:** Analyze the provided video summaries to determine whether the content is indicative of a scam. 

1. **Input**: You will receive a video summary.
2. **Output**: Respond with a JSON object structured as follows:

```json
{
  "is_scam": true | false,
  "confidence": 0.0 ~ 1.0,
  "risk": one of "low, mid, high",
  "evidence": ["evidence1", "evidence2"],
  "explanation": "A detailed explanation of the reasoning behind the determination."
}
```

**Instructions:**
- Evaluate the summary for keywords and phrases that indicate fraudulent intent, such as unrealistic profit claims, urgency statements, or manipulative tactics.
- Classify the video as a scam (`is_scam: true`) or not a scam (`is_scam: false`) based on the characteristics outlined in the summaries.
- Assign a confidence score between 0.0 and 1.0, reflecting your certainty about the classification.
- Determine the level of risk associated with the video content as "low," "mid," or "high."
- Provide specific pieces of evidence that support your conclusion and craft a clear explanation summarizing your analysis.

**Example Input**:
```text
This video discusses unrealistic investment returns and manipulative tactics.
```

**Expected Example Output**:
```json
{
  "is_scam": true,
  "confidence": 0.95,
  "risk": "high",
  "evidence": ["Unrealistic profit claims", "Manipulative tactics"],
  "explanation": "The video makes several unrealistic claims about potential profits and uses urgency to pressure viewers into making hasty decisions."
}
```
"""

In [2]:
import torch
from PIL import Image
from transformers import AutoModel, AutoTokenizer

# load omni model default, the default init_vision/init_audio/init_tts is True
# if load vision-only model, please set init_audio=False and init_tts=False
# if load audio-only model, please set init_vision=False
model = AutoModel.from_pretrained(
    'openbmb/MiniCPM-o-2_6',
    trust_remote_code=True,
    attn_implementation='sdpa', # sdpa or flash_attention_2
    torch_dtype=torch.bfloat16,
    init_vision=True,
    init_audio=True,
    init_tts=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.eval().cuda()
tokenizer = AutoTokenizer.from_pretrained('openbmb/MiniCPM-o-2_6', trust_remote_code=True)

# In addition to vision-only mode, tts processor and vocos also needs to be initialized
model.init_tts()


/home/jaemin/miniconda3/envs/minicpm_0_2_6/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/jaemin/miniconda3/envs/minicpm_0_2_6/lib/python3.10/site-packages/librosa/util/files.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename
/home/jaemin/miniconda3/envs/minicpm_0_2_6/lib/python3.10/site-packages/transformers/models/auto/image_processing_auto.py:513: FutureWarning: The image_processor_class argument is deprecated and will be removed in v4.42. Please use `slow_image_processor_class`, or `fast_image_processor_class` instead
  warnings.warn(
Loading checkpoint

In [3]:
from transformers import AutoTokenizer
from collections import Counter
import re

# 1) 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained("openbmb/MiniCPM-o-2_6", trust_remote_code=True)


# 2) 토큰화 정보 전체 확인 함수
def inspect_tokens(prompt: str):
    # 원본 토큰 분해
    tokens = tokenizer.tokenize(prompt)

    # 토큰 ID (모델이 보는 숫자 시퀀스)
    token_ids = tokenizer(prompt)["input_ids"]

    # 토큰 빈도 (어떤 토큰이 많이 등장하는지)
    freq = Counter(tokens)

    print("=== 1) 전체 토큰 리스트 (순서 그대로) ===")
    print(tokens)

    print("\n=== 2) 토큰 ID 시퀀스 ===")
    print(token_ids)

    print("\n=== 3) 토큰 빈도 (상위 20개) ===")
    for tok, count in freq.most_common(20):
        print(f"{tok} : {count}")

    print("\n=== 4) 토큰 수 ===")
    print(len(token_ids))


# 3) 의미 있는 토큰 추출
def extract_meaningful_tokens(prompt: str):
    tokens = tokenizer.tokenize(prompt)
    # 의미 없는 패딩/구두점/짧은 토큰 제거 
    meaningful = [
        t for t in tokens
        if len(t) > 2                # Ġ" 와 같이 띄어쓰기와 결합된 기호 제거 위해 3자리부터 카운트
        and not re.fullmatch(r"[.,!?;:\-+(){}\[\]]", t)   
        and not re.fullmatch(r"<.*?>", t)                 
    ]
    freq = Counter(meaningful)
    print("의미 있는 토큰 중 빈도 수 정렬")
    for tok, count in freq.most_common(20):
        print(f"{tok} : {count}")




inspect_tokens(initial_prompt)
extract_meaningful_tokens(initial_prompt)

=== 1) 전체 토큰 리스트 (순서 그대로) ===
['**', 'Task', ':**', 'ĠAnaly', 'ze', 'Ġthe', 'Ġprovided', 'Ġvideo', 'Ġsummaries', 'Ġto', 'Ġdetermine', 'Ġwhether', 'Ġthe', 'Ġcontent', 'Ġis', 'Ġindicative', 'Ġof', 'Ġa', 'Ġscam', '.', 'ĠĊĊ', '1', '.', 'Ġ**', 'Input', '**:', 'ĠYou', 'Ġwill', 'Ġreceive', 'Ġa', 'Ġvideo', 'Ġsummary', '.Ċ', '2', '.', 'Ġ**', 'Output', '**:', 'ĠRespond', 'Ġwith', 'Ġa', 'ĠJSON', 'Ġobject', 'Ġstructured', 'Ġas', 'Ġfollows', ':ĊĊ', '```', 'json', 'Ċ', '{Ċ', 'Ġ', 'Ġ"', 'is', '_s', 'cam', '":', 'Ġtrue', 'Ġ|', 'Ġfalse', ',Ċ', 'Ġ', 'Ġ"', 'confidence', '":', 'Ġ', '0', '.', '0', 'Ġ~', 'Ġ', '1', '.', '0', ',Ċ', 'Ġ', 'Ġ"', 'risk', '":', 'Ġone', 'Ġof', 'Ġ"', 'low', ',', 'Ġmid', ',', 'Ġhigh', '",Ċ', 'Ġ', 'Ġ"', 'e', 'vidence', '":', 'Ġ["', 'e', 'vidence', '1', '",', 'Ġ"', 'e', 'vidence', '2', '"],Ċ', 'Ġ', 'Ġ"', 'ex', 'planation', '":', 'Ġ"', 'A', 'Ġdetailed', 'Ġexplanation', 'Ġof', 'Ġthe', 'Ġreasoning', 'Ġbehind', 'Ġthe', 'Ġdetermination', '."Ċ', '}Ċ', '``', '`ĊĊ', '**', 'Instructions', ':', 